In [ ]:
from pathlib import Path

import matplotlib

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path

parent_dir = str(Path.cwd().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from experiment_common import (
    FS,
    DT,
    EXPECTED_SAMPLES,
    CWT_FREQUENCIES,

    ricker_wavelet,
    add_noise,
    cwt_morlet_pywt,
    hos_preprocess_cwt,
    inverse_cwt,
    get_analysis_scales,
)


In [ ]:
dt = DT
length = EXPECTED_SAMPLES * DT
frequency = 100
fs = FS
freqs = CWT_FREQUENCIES
snrs = [-3, -8]

np.random.seed(19)

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 600,
    'savefig.dpi': 600,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8
})

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
row_idx = 0

for snr in snrs:

    t, wavelet = ricker_wavelet(frequency, dt, length)
    noisy = add_noise(wavelet, snr)

    coefficients, freqs_out = cwt_morlet_pywt(noisy, dt, freqs)
    magnitude = np.abs(coefficients)

    filtered, kept_idx = hos_preprocess_cwt(coefficients, freqs_out)
    scales = get_analysis_scales(freqs_out, dt)

    sum_mag = np.sum(magnitude[kept_idx, :], axis=0)

    recon = inverse_cwt(filtered, scales)

    ax = axes[row_idx, 0]
    ax.plot(t, noisy,'k')
    ax.set_title(f'{snr} dB signal')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

    ax = axes[row_idx, 1]
    im = ax.imshow(magnitude, aspect='auto', origin='lower',
                   extent=[t[0], t[-1], freqs_out[0], freqs_out[-1]],
                   cmap='jet')
    ax.set_title('CWT magnitude')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    plt.colorbar(im, ax=ax)

    ax = axes[row_idx, 2]
    ax.plot(t, recon,'k')
    ax.set_title('iCWT reconstruction')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

    row_idx += 1

plt.tight_layout()
plt.show()
